# Qwen3 Text-Only Commercial Campaign SFT - Solo LLM

Notebook dedicado solo al fine-tuning del LLM para generar propuestas comerciales automotrices en JSON.

Este notebook **no** incluye Diffusers, SDXL, DreamBooth, LoRA visual ni generación de imágenes. Usa el dataset corregido de 300 ejemplos:

```text
data/commercial_campaing_sft/commercial_campaign_sft_corrected_300.json
```

Modelo principal:

```python
LLM_BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
```

Fallback manual si la T4 queda corta:

```python
LLM_BASE_MODEL = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit"
```

Flujo recomendado:

1. Ejecutar validación de runtime/GPU.
2. Instalar dependencias limpias.
3. Reiniciar runtime una vez.
4. Ejecutar imports, validación de dataset y baseline.
5. Entrenar LoRA/QLoRA y guardar adapter.
6. Reiniciar runtime para liberar memoria.
7. Evaluar el adapter fine-tuned.
8. Construir reporte comparativo.


## 0. Runtime y GPU

Esta celda no importa `torch`. Solo valida que Colab/Kaggle vea una GPU NVIDIA. Si no aparece GPU, cambia el runtime a GPU antes de entrenar.


In [ ]:
print("Chequeo inicial sin importar torch.")
print("Si vienes de errores de CUDA/NCCL, usa Runtime > Disconnect and delete runtime antes de instalar.")
!nvidia-smi

## 1. Instalar dependencias

Este notebook usa Qwen3 text-only con Unsloth y Transformers 4.x, evitando la ruta Qwen3.5/Transformers v5 que te estaba generando conflictos de CUDA.

Después de ejecutar esta celda, reinicia el runtime una vez y continúa desde la sección 2.


In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    !pip install -q --upgrade pip
    !pip install -q --upgrade --no-cache-dir unsloth unsloth_zoo
    !pip install -q --upgrade --no-cache-dir         "transformers>=4.56.2,<4.57.0"         "datasets>=3.4.1,<4.0.0"         "trl>=0.18.2,<0.25.0,!=0.19.0"         peft accelerate bitsandbytes safetensors matplotlib         "pandas==2.2.2" "scikit-learn>=1.5,<1.9" "cuda-bindings~=12.9.7"
    print("Dependencias instaladas. Reinicia el runtime antes de seguir con la sección 2.")
else:
    print("INSTALL_DEPENDENCIES=False. Se omite instalación.")

## 2. Imports, rutas y configuración

Importante: `unsloth` se importa antes que `torch`, `transformers` o `trl` para que aplique sus parches correctamente.


In [ ]:
import gc
import json
import os
import random
import re
import time
from pathlib import Path

import unsloth
from unsloth import FastModel

from huggingface_hub import hf_hub_download

import torch
import pandas as pd
from IPython.display import display

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)

CANDIDATE_ROOTS = [
    Path.cwd(),
    Path("/content/dmc-tp3-gen-multimodal"),
    Path("/kaggle/working/dmc-tp3-gen-multimodal"),
]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "data").exists()), Path.cwd())
DATA_ROOT = PROJECT_ROOT / "data"
SFT_DIR = DATA_ROOT / "commercial_campaing_sft"
SFT_JSON_PATH = SFT_DIR / "commercial_campaign_sft_corrected_300.json"
SFT_TRAIN_PATH = SFT_DIR / "train_qwen3_text.json"
SFT_EVAL_PATH = SFT_DIR / "eval_qwen3_text.json"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
LLM_OUTPUT_DIR = OUTPUT_ROOT / "qwen3-text-commercial-lora"
EVAL_OUTPUT_DIR = OUTPUT_ROOT / "evaluation"
for path in [OUTPUT_ROOT, LLM_OUTPUT_DIR, EVAL_OUTPUT_DIR, SFT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LLM_BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
LLM_FALLBACK_MODEL = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit"  # fallback manual si falta VRAM

MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = True
LOAD_IN_16BIT = False
TRAIN_FRACTION = 0.8
MAX_EVAL_EXAMPLES = 3
EVAL_MAX_NEW_TOKENS = 200

RUN_BASELINE_EVAL = False
RUN_LLM_TRAINING = False
RUN_FINETUNED_EVAL = False
BUILD_COMPARISON_REPORT = True

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SFT_JSON_PATH:", SFT_JSON_PATH)
print("LLM_BASE_MODEL:", LLM_BASE_MODEL)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, "| VRAM GB:", round(props.total_memory / 1024**3, 2))

## 3. Contrato del dataset SFT

Cada ejemplo debe tener `instruction`, `input` y `output`. El `output` esperado contiene 8 campos comerciales. `image_prompt` forma parte de la salida textual del LLM porque luego sirve como puente hacia el pipeline visual, pero aquí no se entrena ningún diffusion model.


In [ ]:
EXPECTED_SFT_OUTPUT_FIELDS = {
    "strategy",
    "recommended_channel",
    "channel_rationale",
    "channel_plan",
    "ad_copy",
    "image_prompt",
    "kpis",
    "business_note",
}

EXAMPLE_SFT_RECORD = {
    "instruction": "Actua como estratega publicitario para una concesionaria de vehiculos. Genera una propuesta de campana en JSON.",
    "input": "Objetivo: Lead Generation | Vehiculo: SUV hibrida | Rango: mid-range | Audiencia: Families 35-44 | Sector cliente: familias urbanas | Canal historico: Instagram | Ciudad: Miami | Idioma: Spanish | Duracion: 30 Days | Promocion: test drive + financiamiento | ROI: 2.10 | Conversion rate: 0.08 | Engagement: 9",
    "output": {
        "strategy": "Promover seguridad, espacio familiar y ahorro de combustible, cerrando con una invitacion a test drive.",
        "recommended_channel": "Instagram",
        "channel_rationale": "Instagram es visual, permite formularios de leads y encaja con audiencias familiares urbanas.",
        "channel_plan": "Usar Instagram para awareness visual y formularios de lead generation; reforzar con remarketing en Meta Ads.",
        "ad_copy": "Tu familia merece mas espacio, tecnologia y eficiencia. Agenda hoy tu test drive y conoce la SUV hibrida que se adapta a tu ciudad.",
        "image_prompt": "REALCARMODEL real car model in a Spanish Instagram ad for a mid-range hybrid SUV dealership campaign targeting urban families in Miami, bright city background, premium automotive commercial photography, clear space for headline, no readable text",
        "kpis": ["Leads", "Cost per Lead", "Test Drive Bookings", "Conversion Rate", "ROI"],
        "business_note": "Priorizar leads calificados y medir reservas de test drive antes de escalar presupuesto.",
    },
}

print(json.dumps(EXAMPLE_SFT_RECORD, ensure_ascii=False, indent=2))

## 4. Cargar y validar JSON SFT

Valida que el dataset tenga 300 ejemplos y que cada salida incluya los 8 campos esperados.


In [ ]:
def load_sft_records(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"No existe {path}. Copia el JSON corregido de 300 ejemplos en esa ruta.")

    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, list):
        raise ValueError("El dataset SFT debe ser una lista JSON de objetos.")

    validated = []
    errors = []
    for idx, item in enumerate(data):
        if not isinstance(item, dict):
            errors.append((idx, "record is not an object"))
            continue
        for key in ["instruction", "input", "output"]:
            if key not in item:
                errors.append((idx, f"missing {key}"))
        output = item.get("output")
        if isinstance(output, str):
            try:
                output = json.loads(output)
                item = {**item, "output": output}
            except Exception:
                errors.append((idx, "output string is not valid JSON"))
        if not isinstance(item.get("output"), dict):
            errors.append((idx, "output is not an object"))
            continue
        missing = EXPECTED_SFT_OUTPUT_FIELDS - set(item["output"].keys())
        if missing:
            errors.append((idx, f"missing output fields: {sorted(missing)}"))
        validated.append(item)

    if errors:
        preview = errors[:10]
        raise ValueError(f"Errores de validacion: {preview} | total={len(errors)}")

    return validated

sft_records = load_sft_records(SFT_JSON_PATH)
print(f"Registros validados: {len(sft_records)}")
if len(sft_records) != 300:
    print("Aviso: se esperaban 300 ejemplos, pero se encontraron", len(sft_records))
display(pd.DataFrame([{
    "instruction": r["instruction"][:80],
    "input": r["input"][:120],
    "output_fields": ", ".join(sorted(r["output"].keys())),
} for r in sft_records[:3]]))

## 5. Split train/eval y formato instruct

El split se guarda para reproducibilidad. El prompt exige JSON puro para que las métricas estructurales tengan sentido.


In [ ]:
SYSTEM_INSTRUCTION = (
    "Actua como estratega publicitario senior para una concesionaria de vehiculos. "
    "Responde exclusivamente con un objeto JSON valido que tenga exactamente estos campos: "
    "strategy, recommended_channel, channel_rationale, channel_plan, ad_copy, image_prompt, kpis, business_note."
)

def deterministic_split(records, train_fraction=0.8, seed=3407):
    records = list(records)
    rng = random.Random(seed)
    rng.shuffle(records)
    split_idx = int(len(records) * train_fraction)
    return records[:split_idx], records[split_idx:]

def format_sft_text(record):
    response = json.dumps(record["output"], ensure_ascii=False)
    return f"""### Instruction:
{SYSTEM_INSTRUCTION}

### Input:
{record['input']}

### Response:
{response}"""

def build_generation_prompt(record):
    return f"""### Instruction:
{SYSTEM_INSTRUCTION}

### Input:
{record['input']}

### Response:
"""

train_records, eval_records = deterministic_split(sft_records, TRAIN_FRACTION, SEED)
SFT_TRAIN_PATH.write_text(json.dumps(train_records, ensure_ascii=False, indent=2), encoding="utf-8")
SFT_EVAL_PATH.write_text(json.dumps(eval_records, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Train: {len(train_records)} | Eval: {len(eval_records)}")
print("Train path:", SFT_TRAIN_PATH)
print("Eval path:", SFT_EVAL_PATH)
print(format_sft_text(train_records[0])[:1200])

## 6. Helpers de inferencia y métricas

Las métricas son simples y pensadas para comparar estructura, no para reemplazar revisión humana.

- `json_valid`: la respuesta se puede parsear como JSON.
- `field_coverage`: porcentaje de campos obligatorios presentes.
- `recommended_channel_match`: coincidencia exacta del canal recomendado.
- `kpi_jaccard`: overlap entre KPIs esperados y generados.
- `text_jaccard_vs_expected`: similitud textual simple contra la salida esperada.
- `image_prompt_present`: presencia de `image_prompt` no vacío.
- `latency_sec`: tiempo de generación.


In [ ]:
def extract_json_object(text):
    if isinstance(text, dict):
        return text
    if not isinstance(text, str):
        return None
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None

def tokens_for_jaccard(value):
    return set(re.findall(r"\w+", str(value).lower()))

def jaccard(a, b):
    aw = tokens_for_jaccard(a)
    bw = tokens_for_jaccard(b)
    if not aw and not bw:
        return 1.0
    return len(aw & bw) / max(1, len(aw | bw))

def list_jaccard(a, b):
    aset = {str(x).strip().lower() for x in (a or [])}
    bset = {str(x).strip().lower() for x in (b or [])}
    if not aset and not bset:
        return 1.0
    return len(aset & bset) / max(1, len(aset | bset))

def score_generated_record(item):
    parsed = extract_json_object(item.get("generated", ""))
    expected = item.get("expected", {})
    json_valid = parsed is not None
    if not parsed:
        parsed = {}
    expected_channel = str(expected.get("recommended_channel", "")).strip().lower()
    generated_channel = str(parsed.get("recommended_channel", "")).strip().lower()
    field_coverage = len(EXPECTED_SFT_OUTPUT_FIELDS & set(parsed.keys())) / len(EXPECTED_SFT_OUTPUT_FIELDS)
    expected_text = json.dumps(expected, ensure_ascii=False, sort_keys=True)
    generated_text = json.dumps(parsed, ensure_ascii=False, sort_keys=True)
    return {
        "example_id": item.get("example_id"),
        "model": item.get("model"),
        "json_valid": float(json_valid),
        "field_coverage": field_coverage,
        "recommended_channel_match": float(bool(expected_channel) and expected_channel == generated_channel),
        "kpi_jaccard": list_jaccard(expected.get("kpis"), parsed.get("kpis")),
        "text_jaccard_vs_expected": jaccard(expected_text, generated_text),
        "image_prompt_present": float(bool(str(parsed.get("image_prompt", "")).strip())),
        "latency_sec": item.get("latency_sec"),
    }

def score_llm_outputs(outputs):
    return pd.DataFrame([score_generated_record(item) for item in outputs])

def cleanup_model(model=None, tokenizer=None):
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

## 7. Carga de Qwen3 text-only y generación secuencial

Se usa `FastModel`, carga 4-bit y generación manual con `model.generate`. Esto evita mezclar `pipeline` con modelos parcheados por Unsloth.


In [ ]:
def validate_hf_config_available(model_name_or_path):
    model_ref = str(model_name_or_path)
    if Path(model_ref).exists() or "/" not in model_ref:
        return
    try:
        hf_hub_download(repo_id=model_ref, filename="config.json")
    except Exception as exc:
        raise RuntimeError(
            f"No se pudo descargar config.json de {model_ref}. "
            "Verifica conexión a Hugging Face, nombre del modelo y permisos del runtime."
        ) from exc

def load_qwen3_for_generation(model_name_or_path):
    validate_hf_config_available(model_name_or_path)
    try:
        model, tokenizer = FastModel.from_pretrained(
            model_name=str(model_name_or_path),
            max_seq_length=MAX_SEQ_LENGTH,
            load_in_4bit=LOAD_IN_4BIT,
            load_in_16bit=LOAD_IN_16BIT,
            fast_inference=False,
        )
    except Exception as exc:
        raise RuntimeError(
            f"No se pudo cargar {model_name_or_path} con FastModel. "
            "Verifica que el runtime este limpio, que hayas reiniciado tras instalar dependencias "
            "y que el modelo exista en Hugging Face. Si falta VRAM, cambia manualmente "
            "LLM_BASE_MODEL por LLM_FALLBACK_MODEL en la sección 2."
        ) from exc

    FastModel.for_inference(model)
    if getattr(tokenizer, "pad_token_id", None) is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

def generate_one(model, tokenizer, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def run_generation_phase(model_label, model_name_or_path, records, output_path):
    model, tokenizer = load_qwen3_for_generation(model_name_or_path)
    results = []
    for idx, record in enumerate(records):
        prompt = build_generation_prompt(record)
        start = time.perf_counter()
        generated = generate_one(model, tokenizer, prompt, EVAL_MAX_NEW_TOKENS)
        latency = time.perf_counter() - start
        results.append({
            "example_id": idx,
            "model": model_label,
            "input": record["input"],
            "expected": record["output"],
            "generated": generated,
            "latency_sec": latency,
        })
        print(f"{model_label} example {idx}: {latency:.2f}s")
    output_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Outputs guardados en:", output_path)
    cleanup_model(model, tokenizer)
    return results

## 8. Baseline del modelo no tuneado

Ejecuta esta sección antes del entrenamiento para guardar respuestas del modelo base. Si la memoria queda ocupada, reinicia el runtime antes de entrenar.


In [ ]:
BASE_OUTPUT_PATH = EVAL_OUTPUT_DIR / "qwen3_text_base_outputs.json"

if RUN_BASELINE_EVAL:
    if not torch.cuda.is_available():
        raise RuntimeError("Se recomienda GPU para evaluar Qwen3 text-only.")
    eval_subset = eval_records[:MAX_EVAL_EXAMPLES]
    base_outputs = run_generation_phase("base", LLM_BASE_MODEL, eval_subset, BASE_OUTPUT_PATH)
    base_metrics = score_llm_outputs(base_outputs)
    display(base_metrics)
else:
    print("RUN_BASELINE_EVAL=False. Se omite baseline del modelo base.")
    print("Output esperado:", BASE_OUTPUT_PATH)

## 9. Fine-tuning LoRA/QLoRA con Qwen3 text-only

Esta celda entrena adapters LoRA sobre atención completa + MLP y guarda el resultado en `outputs/qwen3-text-commercial-lora/`.

Perfil conservador para T4:

- 4-bit (`LOAD_IN_4BIT=True`)
- batch size 1
- gradient accumulation 8
- 1 epoch
- LoRA rank 8


In [ ]:
if RUN_LLM_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Se requiere GPU para entrenar Qwen3 text-only con LoRA.")

    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer

    validate_hf_config_available(LLM_BASE_MODEL)
    model, tokenizer = FastModel.from_pretrained(
        model_name=LLM_BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        load_in_16bit=LOAD_IN_16BIT,
        fast_inference=False,
    )

    model = FastModel.get_peft_model(
        model,
        r=8,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        max_seq_length=MAX_SEQ_LENGTH,
    )

    train_dataset = Dataset.from_list([{"text": format_sft_text(r)} for r in train_records])
    eval_dataset = Dataset.from_list([{"text": format_sft_text(r)} for r in eval_records]) if eval_records else None

    training_args = SFTConfig(
        output_dir=str(LLM_OUTPUT_DIR),
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=5,
        save_strategy="epoch",
        eval_strategy="no",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        seed=SEED,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
    )

    trainer.train()
    model.save_pretrained(str(LLM_OUTPUT_DIR))
    tokenizer.save_pretrained(str(LLM_OUTPUT_DIR))
    print("Adapter guardado en:", LLM_OUTPUT_DIR)
    cleanup_model(model, tokenizer)
else:
    print("RUN_LLM_TRAINING=False. Se omite entrenamiento.")
    print("Adapter esperado:", LLM_OUTPUT_DIR)

## 10. Evaluación del modelo fine-tuned

Recomendación: después de entrenar y guardar el adapter, reinicia el runtime y ejecuta desde la sección 2 hasta esta celda con `RUN_FINETUNED_EVAL=True`.


In [ ]:
FINETUNED_OUTPUT_PATH = EVAL_OUTPUT_DIR / "qwen3_text_finetuned_outputs.json"

if RUN_FINETUNED_EVAL:
    if not any(LLM_OUTPUT_DIR.glob("*")):
        raise RuntimeError(f"No se encontro adapter en {LLM_OUTPUT_DIR}. Ejecuta entrenamiento primero.")
    if not torch.cuda.is_available():
        raise RuntimeError("Se recomienda GPU para evaluar el adapter fine-tuned.")
    eval_subset = eval_records[:MAX_EVAL_EXAMPLES]
    ft_outputs = run_generation_phase("fine_tuned", LLM_OUTPUT_DIR, eval_subset, FINETUNED_OUTPUT_PATH)
    ft_metrics = score_llm_outputs(ft_outputs)
    display(ft_metrics)
else:
    print("RUN_FINETUNED_EVAL=False. Se omite evaluación fine-tuned.")
    print("Output esperado:", FINETUNED_OUTPUT_PATH)

## 11. Reporte comparativo base vs fine-tuned

Esta sección combina los outputs guardados y produce métricas agregadas y una tabla lado a lado.


In [ ]:
METRICS_PATH = EVAL_OUTPUT_DIR / "qwen3_text_llm_metrics.csv"
COMPARISON_TABLE_PATH = EVAL_OUTPUT_DIR / "qwen3_text_comparison_table.csv"

if BUILD_COMPARISON_REPORT:
    all_outputs = []
    for path in [BASE_OUTPUT_PATH, FINETUNED_OUTPUT_PATH]:
        if path.exists():
            all_outputs.extend(json.loads(path.read_text(encoding="utf-8")))
        else:
            print("No existe aun:", path)

    if not all_outputs:
        print("No hay outputs para comparar. Ejecuta baseline y/o fine-tuned eval primero.")
    else:
        metrics_df = score_llm_outputs(all_outputs)
        metrics_df.to_csv(METRICS_PATH, index=False)
        display(metrics_df)

        numeric_cols = [
            "json_valid",
            "field_coverage",
            "recommended_channel_match",
            "kpi_jaccard",
            "text_jaccard_vs_expected",
            "image_prompt_present",
            "latency_sec",
        ]
        summary_df = metrics_df.groupby("model")[numeric_cols].mean(numeric_only=True).reset_index()
        display(summary_df)

        by_model = {}
        for item in all_outputs:
            by_model[(item["example_id"], item["model"])] = item

        table_rows = []
        for idx, record in enumerate(eval_records[:MAX_EVAL_EXAMPLES]):
            base_item = by_model.get((idx, "base"), {})
            ft_item = by_model.get((idx, "fine_tuned"), {})
            table_rows.append({
                "example_id": idx,
                "input": record["input"],
                "expected": json.dumps(record["output"], ensure_ascii=False),
                "base_generated": base_item.get("generated", ""),
                "fine_tuned_generated": ft_item.get("generated", ""),
            })

        comparison_df = pd.DataFrame(table_rows)
        comparison_df.to_csv(COMPARISON_TABLE_PATH, index=False)
        display(comparison_df)
        print("Metricas guardadas en:", METRICS_PATH)
        print("Tabla comparativa guardada en:", COMPARISON_TABLE_PATH)
else:
    print("BUILD_COMPARISON_REPORT=False. Se omite reporte comparativo.")

## 12. Notas de lectura de resultados

Una mejora después del fine-tuning debería verse principalmente en:

- más respuestas JSON válidas;
- mayor cobertura de campos;
- `recommended_channel` más alineado con el esperado;
- KPIs más parecidos al dataset;
- `image_prompt` presente y útil para integración visual posterior;
- menor necesidad de reparación manual de formato.

Estas métricas son indicativas. Para decidir calidad final, revisa también ejemplos lado a lado.

Si Qwen3-4B queda pesado en T4, cambia manualmente en la sección 2:

```python
LLM_BASE_MODEL = LLM_FALLBACK_MODEL
```

El fallback mantiene el flujo text-only y evita volver al camino multimodal.
